In [1]:
from catboost import CatBoostClassifier, CatBoostRegressor

# Load classification model
edge_classifier = CatBoostClassifier()
edge_classifier.load_model(
    "../models/catboost_rit_39_features.cbm"
)

# Load regression model
edge_regressor = CatBoostRegressor()
edge_regressor.load_model(
    "../models/catboost_regression_39_features.cbm"
)

print("Classification model features:",
      edge_classifier.n_features_in_)

print("Regression model features:",
      edge_regressor.n_features_in_)

print("Classification classes:",
      edge_classifier.classes_)

print("Both models loaded successfully!")

Classification model features: 0
Regression model features: 0
Classification classes: [0 1 2 3 4]
Both models loaded successfully!


In [3]:
print("Classification features:",
      len(edge_classifier.feature_names_))

print("Regression features:",
      len(edge_regressor.feature_names_))

print("\nClassification tree count:",
      edge_classifier.tree_count_)

print("Regression tree count:",
      edge_regressor.tree_count_)

Classification features: 39
Regression features: 39

Classification tree count: 242
Regression tree count: 192


In [4]:
import pandas as pd
import json
import numpy as np
import time
import os

# Load the 39 feature names
with open("../models/rit_feature_list.json", "r") as f:
    rit_feature_cols = json.load(f)

# Load the processed dataset
data = pd.read_csv(
    "../data/processed/model_table_5min.csv"
)

# Recreate the model input
X_rit = data[rit_feature_cols].copy()

# Select one sample
sample_input = X_rit.iloc[[0]]

print("Feature count:", len(rit_feature_cols))
print("Sample shape:", sample_input.shape)
display(sample_input)

Feature count: 39
Sample shape: (1, 39)


,occ,occ_lag1,occ_lag2,occ_lag3,occ_lag6,occ_ratio,co2,co2_lag1,co2_lag2,co2_lag3,...,illuminance_lag2,illuminance_lag3,illuminance_lag6,illuminance_age,hour_sin,hour_cos,month,room,capacity,is_lecture
0,0.0,0.0,0.0,0.0,0.0,0.0,482.880005,476.799988,483.839996,475.839996,...,0.0,0.0,0.0,34.0,0.130526,0.991445,9,1,84,1


In [5]:
def measure_inference(model, sample, runs=100):

    # Warm-up
    for _ in range(10):
        model.predict(sample)

    times = []

    for _ in range(runs):
        start = time.perf_counter()
        model.predict(sample)
        end = time.perf_counter()

        times.append((end - start) * 1000)

    return {
        "Average (ms)": round(np.mean(times), 4),
        "Median (ms)": round(np.median(times), 4),
        "P95 (ms)": round(np.percentile(times, 95), 4)
    }


classification_speed = measure_inference(
    edge_classifier, sample_input
)

regression_speed = measure_inference(
    edge_regressor, sample_input
)

display(pd.DataFrame(
    [classification_speed, regression_speed],
    index=["Classification", "Regression"]
))

,Average (ms),Median (ms),P95 (ms)
Classification,1.3111,1.2969,1.7008
Regression,1.5040,1.4568,1.8148


In [6]:
import os

classification_size = os.path.getsize(
    "../models/catboost_rit_39_features.cbm"
) / (1024 * 1024)

regression_size = os.path.getsize(
    "../models/catboost_regression_39_features.cbm"
) / (1024 * 1024)

model_sizes = pd.DataFrame({
    "Model": ["Classification", "Regression"],
    "Size (MB)": [
        round(classification_size, 4),
        round(regression_size, 4)
    ]
})

display(model_sizes)

,Model,Size (MB)
0,Classification,0.7758
1,Regression,0.2250


In [7]:
edge_results = pd.DataFrame([
    {
        "Model": "Classification",
        "Features": 39,
        "Trees": 242,
        "Size_MB": classification_size,
        "Average_Latency_ms": classification_speed["Average (ms)"],
        "Median_Latency_ms": classification_speed["Median (ms)"],
        "P95_Latency_ms": classification_speed["P95 (ms)"]
    },
    {
        "Model": "Regression",
        "Features": 39,
        "Trees": 192,
        "Size_MB": regression_size,
        "Average_Latency_ms": regression_speed["Average (ms)"],
        "Median_Latency_ms": regression_speed["Median (ms)"],
        "P95_Latency_ms": regression_speed["P95 (ms)"]
    }
])

edge_results.to_csv(
    "../results/edge_inference_comparison.csv",
    index=False
)

display(edge_results)

print("\nEdge inference results saved!")

,Model,Features,Trees,Size_MB,Average_Latency_ms,Median_Latency_ms,P95_Latency_ms
0,Classification,39,242,0.775772,1.3111,1.2969,1.7008
1,Regression,39,192,0.225006,1.5040,1.4568,1.8148



Edge inference results saved!
